# Линейная алгебра для Data Science
## Урок 2 — Скалярное произведение, косинусная близость и нормализация

На прошлом уроке вектор описывался длиной и координатами. Теперь введём операцию, которая
связывает **два** вектора и измеряет, насколько они *сонаправлены*, — **скалярное
произведение**. Из него вырастут угол между наблюдениями, косинусная близость и
нормализация, а в конце мы увидим, что ровно эта операция лежит в основе механизма
внимания (attention) в больших языковых моделях.

Продолжаем работать с датасетом `Davis` (вес и рост 200 человек).


In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset

dataset = load_dataset('aiacademy-kg/davis_dataset', split='train')
df = dataset.to_pandas()

In [ ]:
df.head()

In [ ]:
#@title Оформление и хелперы визуализации — запустите первой { display-mode: "form" }
from IPython.display import HTML, display

CSS = r"""
.vec-root{--bg:#12161c;--surface:#1a1f27;--ink:#e6e9ec;--soft:#a7b0ba;--line:#2a313b;
  --green:#5fd08a;--blue:#6ab0f3;--amber:#e0b25a;--rose:#e0757f;
  font-family:Georgia,'Times New Roman',serif;color:var(--ink);
  background:var(--bg);border-radius:14px;padding:2px 0;}
.vec-root .wrap{max-width:820px;margin:0 auto;padding:18px 22px;font-size:17px;line-height:1.65;}
.vec-root h2{font-family:-apple-system,'Segoe UI',sans-serif;font-size:1.5rem;color:#fff;
  border-top:2px solid var(--line);padding-top:.5em;margin:1.4em 0 .5em;}
.vec-root h3{font-family:sans-serif;color:var(--green);font-size:1.15rem;margin:1.3em 0 .3em;}
.vec-root p{margin:.6em 0;} .vec-root b,.vec-root strong{color:#fff;}
.vec-root em{color:var(--soft);} .vec-root ul,.vec-root ol{padding-left:1.4em;} .vec-root li{margin:.28em 0;}
.vec-root a{color:var(--green);}
.vec-root .box{border-radius:12px;padding:14px 18px;margin:1.2em 0;font-family:sans-serif;
  font-size:.95rem;border:1px solid var(--line);background:var(--surface);}
.vec-root .box .t{font-weight:700;font-size:.8rem;letter-spacing:.06em;text-transform:uppercase;margin-bottom:5px;}
.vec-root .box.def{background:#15251d;border-color:#27503a;} .vec-root .box.def .t{color:var(--green);}
.vec-root .box.idea{background:#14202e;border-color:#274a63;} .vec-root .box.idea .t{color:var(--blue);}
.vec-root .box.warn{background:#241f14;border-color:#5a4a27;} .vec-root .box.warn .t{color:var(--amber);}
.vec-root .box.take{background:#241419;border-color:#5a2730;} .vec-root .box.take .t{color:var(--rose);}
.vec-root .widget{border:1px solid var(--line);border-radius:12px;background:var(--surface);
  padding:16px;margin:1.4em 0;}
.vec-root .widget-title{font-family:sans-serif;font-weight:700;font-size:.95rem;margin-bottom:2px;color:#fff;}
.vec-root .widget-sub{font-family:sans-serif;font-size:.83rem;color:var(--soft);margin-bottom:10px;}
.vec-root .widget>svg{display:block;width:100%;height:auto;background:#0e1319;border-radius:10px;}
.vec-root .controls{display:flex;flex-wrap:wrap;gap:12px 20px;align-items:center;margin-top:12px;
  font-family:sans-serif;font-size:.88rem;color:var(--soft);}
.vec-root .control{display:flex;align-items:center;gap:8px;}
.vec-root input[type=range]{accent-color:var(--green);width:150px;}
.vec-root .val{font-family:'SF Mono',Menlo,monospace;font-size:.82rem;color:var(--green);min-width:40px;}
.vec-root .btn{font-family:sans-serif;font-size:.85rem;font-weight:600;border:1px solid var(--green);
  background:transparent;color:var(--green);padding:6px 14px;border-radius:8px;cursor:pointer;}
.vec-root .btn:hover{background:var(--green);color:#0e1319;}
.vec-root .readout{font-family:'SF Mono',Menlo,monospace;font-size:.84rem;background:#0e1319;
  padding:9px 12px;border-radius:8px;margin-top:12px;color:var(--soft);}
.vec-root .gridline{stroke:#20272f;stroke-width:1;} .vec-root .axis{stroke:#3a434f;stroke-width:1.2;}
.vec-root .tick-label{font-family:sans-serif;font-size:11px;fill:#6b7580;}
.vec-root code{font-family:Menlo,monospace;font-size:.86em;background:#0e1319;color:var(--amber);
  padding:1px 6px;border-radius:5px;}
.vec-root details{border:1px dashed #38414c;border-radius:10px;padding:4px 16px;margin:1.1em 0;background:#161b22;}
.vec-root summary{cursor:pointer;font-family:sans-serif;font-weight:700;color:var(--blue);padding:8px 0;}
.vec-root table{border-collapse:collapse;width:100%;margin:1em 0;font-family:sans-serif;font-size:.9rem;}
.vec-root th,.vec-root td{border:1px solid var(--line);padding:7px 10px;text-align:left;}
.vec-root td{color:#fff;}
.vec-root th{background:#0e1319;color:var(--green);font-weight:700;}
.vec-root tr:nth-child(even) td{background:#151a21;}
"""

JS = r"""
if(!window.__VEC_READY__){window.__VEC_READY__=true;
const NS="http://www.w3.org/2000/svg";
window.scale=(d,r)=>{const[a,b]=d,[c,e]=r;return v=>c+(v-a)*(e-c)/(b-a);};
window.el=(t,at={})=>{const x=document.createElementNS(NS,t);for(const k in at)x.setAttribute(k,at[k]);return x;};
window.clamp=(v,lo,hi)=>Math.max(lo,Math.min(hi,v));
window.arrow=(x1,y1,x2,y2,o={})=>{const g=el("g");const c=o.color||"#5fd08a",w=o.width||2.6;
  const ln={x1,y1,x2,y2,stroke:c,"stroke-width":w,"stroke-linecap":"round"};if(o.dash)ln["stroke-dasharray"]=o.dash;
  g.appendChild(el("line",ln));const L=Math.hypot(x2-x1,y2-y1);
  if(L>4){const an=Math.atan2(y2-y1,x2-x1),hl=o.head||12,ha=.42;
   const p1x=x2-hl*Math.cos(an-ha),p1y=y2-hl*Math.sin(an-ha),p2x=x2-hl*Math.cos(an+ha),p2y=y2-hl*Math.sin(an+ha);
   g.appendChild(el("polygon",{points:`${x2},${y2} ${p1x},${p1y} ${p2x},${p2y}`,fill:c}));}return g;};
window.svgPoint=(svg,ev)=>{const p=svg.createSVGPoint();const s=(ev.touches&&ev.touches.length)?ev.touches[0]:ev;
  p.x=s.clientX;p.y=s.clientY;return p.matrixTransform(svg.getScreenCTM().inverse());};
window.onDrag=(svg,h,cb)=>{let on=false;const st=e=>{on=true;e.preventDefault();};
  const mv=e=>{if(!on)return;e.preventDefault();cb(svgPoint(svg,e));};const en=()=>on=false;
  h.style.cursor="grab";h.addEventListener("mousedown",st);h.addEventListener("touchstart",st,{passive:false});
  window.addEventListener("mousemove",mv);window.addEventListener("touchmove",mv,{passive:false});
  window.addEventListener("mouseup",en);window.addEventListener("touchend",en);};
window.drawFrame=(g,c)=>{const{x,y,x0,x1,y0,y1,xt=[],yt=[]}=c;
  for(const tx of xt){g.appendChild(el("line",{x1:x(tx),y1:y(y0),x2:x(tx),y2:y(y1),class:"gridline"}));
    const t=el("text",{x:x(tx),y:y(y0)+15,class:"tick-label","text-anchor":"middle"});t.textContent=tx;g.appendChild(t);}
  for(const ty of yt){g.appendChild(el("line",{x1:x(x0),y1:y(ty),x2:x(x1),y2:y(ty),class:"gridline"}));
    const t=el("text",{x:x(x0)-8,y:y(ty)+4,class:"tick-label","text-anchor":"end"});t.textContent=ty;g.appendChild(t);}
  const ax=(y0<=0&&y1>=0)?0:y0,ay=(x0<=0&&x1>=0)?0:x0;
  g.appendChild(el("line",{x1:x(x0),y1:y(ax),x2:x(x1),y2:y(ax),class:"axis"}));
  g.appendChild(el("line",{x1:x(ay),y1:y(y0),x2:x(ay),y2:y(y1),class:"axis"}));};
const W=600,H=380,PL=36,PR=16,PT=16,PB=28;
window.X0=-6;window.X1=6;window.Y0=-3.7;window.Y1=3.7;
window.SX=scale([X0,X1],[PL,W-PR]);window.SY=scale([Y0,Y1],[H-PB,PT]);
window.IX=scale([PL,W-PR],[X0,X1]);window.IY=scale([H-PB,PT],[Y0,Y1]);
window.OX=SX(0);window.OY=SY(0);window.XT=[-6,-4,-2,2,4,6];window.YT=[-3,-1,1,3];
window.COL={a:"#6ab0f3",b:"#e0b25a",sum:"#e0757f",v:"#6ab0f3",res:"#5fd08a"};
window.freshFrame=(svg)=>{while(svg.firstChild)svg.removeChild(svg.firstChild);
  const g=el("g");svg.appendChild(g);drawFrame(g,{x:SX,y:SY,x0:X0,x1:X1,y0:Y0,y1:Y1,xt:XT,yt:YT});return g;};
window.handle=(g,dx,dy,color,onMove,svg)=>{const c=el("circle",{cx:SX(dx),cy:SY(dy),r:9,
  fill:color,"fill-opacity":.25,stroke:color,"stroke-width":2});g.appendChild(c);
  onDrag(svg,c,p=>onMove(clamp(IX(p.x),X0,X1),clamp(IY(p.y),Y0,Y1)));return c;};
window.label=(g,dx,dy,text,color)=>{const t=el("text",{x:SX(dx),y:SY(dy),fill:color,
  "font-family":"-apple-system,Segoe UI,sans-serif","font-size":15,"font-weight":700});
  t.textContent=text;g.appendChild(t);return t;};
// --- local SQUARE plane (equal px per unit): needed for honest circles/angles ---
window.plane=(half)=>{const aw=W-PL-PR,ah=H-PT-PB,ppu=Math.min(aw,ah)/(2*half);
  const cx=PL+aw/2,cy=PT+ah/2;
  const sx=v=>cx+v*ppu,sy=v=>cy-v*ppu,ix=px=>(px-cx)/ppu,iy=py=>(cy-py)/ppu;
  const ticks=[];for(let t=-Math.floor(half);t<=Math.floor(half);t++)if(t!==0)ticks.push(t);
  return {sx,sy,ix,iy,ox:sx(0),oy:sy(0),ppu,ticks,half,
    frame(g){drawFrame(g,{x:sx,y:sy,x0:-half,x1:half,y0:-half,y1:half,xt:ticks,yt:ticks});},
    drag(svg,g,dx,dy,color,cb){const c=el("circle",{cx:sx(dx),cy:sy(dy),r:9,fill:color,
      "fill-opacity":.25,stroke:color,"stroke-width":2});g.appendChild(c);
      onDrag(svg,c,p=>cb(clamp(ix(p.x),-half,half),clamp(iy(p.y),-half,half)));return c;}};};
}
"""

MJ_CFG = "<script>window.MathJax=window.MathJax||{tex:{inlineMath:[['$','$']],displayMath:[['$$','$$']]},svg:{fontCache:'global'}};</script>"
MJ = '<script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-svg.js"></script>'

def viz(body, script=""):
    """Wrap an HTML body with the dark theme, JS helpers and MathJax. Self-contained (Colab)."""
    head = "<style>" + CSS + "</style>"
    tail = ("<script>" + JS + "\n" + script +
            "\nif(window.MathJax&&MathJax.typesetPromise){MathJax.typesetPromise();}</script>")
    return head + '<div class="vec-root"><div class="wrap">' + body + "</div></div>" + MJ_CFG + MJ + tail

print("Оформление загружено. Функция viz() готова.")

In [ ]:
#@title Теория 1 — Скалярное произведение { display-mode: "form" }
body = r"""
<h2>1. Скалярное произведение</h2>

<p><b>Скалярным произведением</b> векторов $a=(a_1,\ldots,a_n)$ и $b=(b_1,\ldots,b_n)$
называется число
$$ \langle a,b\rangle \;=\; a_1b_1+a_2b_2+\cdots+a_nb_n \;=\; \sum_{i=1}^{n} a_ib_i. $$
Наряду с $\langle a,b\rangle$ употребляют записи $a\cdot b$ и $a^{\top}b$. Результат —
именно <b>скаляр</b> (число), а не вектор; отсюда и название.</p>

<div class="widget"><div class="widget-title">Строка на столбец: перемножаем одноимённые координаты и складываем</div>
<div class="widget-sub">В записи $a^{\top}b$ вектор-строка $a^{\top}$ умножается на вектор-столбец $b$.
Стрелки соединяют координаты с одинаковыми номерами: $a_1$ с $b_1$, $a_2$ с $b_2$, и так далее.</div>
<svg viewBox="0 0 600 250" id="svg-dotmul"></svg></div>

<div class="box def"><div class="t">Геометрическая форма</div>
Скалярное произведение выражается через длины векторов и угол $\varphi$ между ними:
$$ \langle a,b\rangle = \lVert a\rVert\,\lVert b\rVert\cos\varphi. $$
Отсюда его знак несёт геометрический смысл: $\langle a,b\rangle>0$ при остром угле,
$\langle a,b\rangle=0$ при $\varphi=90^\circ$ (векторы <b>ортогональны</b>),
$\langle a,b\rangle<0$ при тупом угле.</p></div>

<p>Две формы согласованы: алгебраическая (сумма произведений координат) удобна для
вычислений, геометрическая ($\lVert a\rVert\lVert b\rVert\cos\varphi$) — для истолкования.
В частности, при $a=b$ получаем связь с длиной из прошлого урока:
$$ \langle a,a\rangle = \lVert a\rVert^2. $$</p>

<h3>Проекция одного вектора на другой</h3>
<p>Скалярное произведение измеряет, какая часть вектора $a$ приходится на направление $b$.
<b>Скалярная проекция</b> вектора $a$ на направление $b$ — это число со знаком
$$ \operatorname{pr}_b a = \frac{\langle a,b\rangle}{\lVert b\rVert}=\lVert a\rVert\cos\varphi, $$
положительное при остром угле и отрицательное при тупом; его модуль равен длине проекции.
Соответствующий <b>вектор-проекция</b> равен $\dfrac{\langle a,b\rangle}{\lVert b\rVert^2}\,b$.
На графике ниже проекция показана точкой на прямой $b$; перетаскивайте концы векторов.</p>

<div class="widget"><div class="widget-title">Скалярное произведение и проекция</div>
<div class="widget-sub">Перетащите концы $a$ (синий) и $b$ (янтарный). Зелёным — проекция
$a$ на $b$, пунктиром — перпендикуляр.</div>
<svg viewBox="0 0 600 380" id="svg-dot"></svg>
<div class="readout" id="dot-ro"></div></div>

<div class="box idea"><div class="t">Связь с физикой</div>
В школьном курсе физики <b>работа</b> постоянной силы $\vec F$ на перемещении $\vec s$
равна $A=\lVert\vec F\rVert\,\lVert\vec s\rVert\cos\varphi$ — это и есть скалярное
произведение $\langle \vec F,\vec s\rangle$. Сила, направленная поперёк движения
($\varphi=90^\circ$), работы не совершает: проекция равна нулю. Тот же самый механизм —
«сколько одного направления содержится в другом» — оказывается центральным и в машинном
обучении.</div>
"""
script = r"""
(function(){const svg=document.getElementById("svg-dotmul");if(!svg)return;
 while(svg.firstChild)svg.removeChild(svg.firstChild);const g=el("g");svg.appendChild(g);
 const cols=["#6ab0f3","#e0b25a","#5fd08a"],A=["a₁","a₂","a₃"],B=["b₁","b₂","b₃"];
 function box(x,y,w,h,s,c){g.appendChild(el("rect",{x,y,width:w,height:h,rx:6,fill:"#0e1319",stroke:c,"stroke-width":1.7}));
  const t=el("text",{x:x+w/2,y:y+h/2+6,fill:"#fff","text-anchor":"middle","font-family":"SF Mono,Menlo,monospace","font-size":18});t.textContent=s;g.appendChild(t);}
 function txt(x,y,s,c,sz,mono){const t=el("text",{x,y,fill:c||"#a7b0ba","font-family":(mono?"SF Mono,Menlo,monospace":"-apple-system,Segoe UI,sans-serif"),"font-size":sz||18,"font-weight":700});t.textContent=s;g.appendChild(t);}
 function brk(x,y,h,d){g.appendChild(el("path",{d:`M${x} ${y} h${d*10} v${h} h${-d*10}`,fill:"none",stroke:"#5a636d","stroke-width":2}));}
 // curved connector a_i -> b_i, routed BELOW the boxes (never crosses the numbers)
 function link(x1,y1,x2,y2,c){const cy=(y1+y2)/2+18;
  g.appendChild(el("path",{d:`M${x1} ${y1} C ${x1} ${cy}, ${x2-58} ${y2}, ${x2} ${y2}`,fill:"none",stroke:c,"stroke-width":2}));
  g.appendChild(el("polygon",{points:`${x2},${y2} ${x2-9},${y2-5} ${x2-9},${y2+5}`,fill:c}));}
 // row vector aᵀ (top)
 txt(20,64,"aᵀ =");const rx=[70,124,178],rw=46,rh=34,ry=40;
 brk(64,36,42,-1);brk(230,36,42,1);
 // column vector b (right, vertically centred)
 txt(296,127,"b =");const cbx=337,cby=[52,110,168],cw=46,ch=34;
 brk(331,48,154,-1);brk(389,48,154,1);
 // connectors first, boxes on top so numbers stay readable
 for(let i=0;i<3;i++)link(rx[i]+rw/2,ry+rh,cbx,cby[i]+ch/2,cols[i]);
 A.forEach((s,i)=>box(rx[i],ry,rw,rh,s,cols[i]));
 B.forEach((s,i)=>box(cbx,cby[i],cw,ch,s,cols[i]));
 // result line
 txt(20,232,"⟨a, b⟩ = aᵀb =");
 txt(196,232,"a₁b₁",cols[0],18,true);txt(250,232,"+");txt(268,232,"a₂b₂",cols[1],18,true);
 txt(322,232,"+");txt(340,232,"a₃b₃",cols[2],18,true);
})();
(function(){const svg=document.getElementById("svg-dot");if(!svg)return;
 const ro=document.getElementById("dot-ro");let a={x:3.4,y:2.2},b={x:4.2,y:-0.6};
 function R(){const g=freshFrame(svg);
  const bb=b.x*b.x+b.y*b.y, dot=a.x*b.x+a.y*b.y, t=dot/bb;
  const pr={x:t*b.x,y:t*b.y};
  // guide line along b
  const k=6/Math.max(Math.abs(b.x),Math.abs(b.y),.001);
  g.appendChild(el("line",{x1:SX(-k*b.x),y1:SY(-k*b.y),x2:SX(k*b.x),y2:SY(k*b.y),stroke:"#232a33","stroke-width":1.4}));
  // perpendicular from tip of a to projection
  g.appendChild(el("line",{x1:SX(a.x),y1:SY(a.y),x2:SX(pr.x),y2:SY(pr.y),stroke:"#5fd08a","stroke-width":1.6,"stroke-dasharray":"5 4"}));
  g.appendChild(arrow(OX,OY,SX(b.x),SY(b.y),{color:COL.b}));
  g.appendChild(arrow(OX,OY,SX(a.x),SY(a.y),{color:COL.a}));
  g.appendChild(el("circle",{cx:SX(pr.x),cy:SY(pr.y),r:5,fill:"#5fd08a"}));
  handle(g,a.x,a.y,COL.a,(x,y)=>{a={x,y};R();},svg);
  handle(g,b.x,b.y,COL.b,(x,y)=>{b={x,y};R();},svg);
  label(g,a.x+.2,a.y+.3,"a",COL.a);label(g,b.x+.2,b.y+.3,"b",COL.b);
  const na=Math.hypot(a.x,a.y),nb=Math.hypot(b.x,b.y);
  const cos=dot/(na*nb),ang=Math.acos(Math.max(-1,Math.min(1,cos)))*180/Math.PI;
  const sign=dot>0.001?"острый угол":dot<-0.001?"тупой угол":"ортогональны";
  ro.innerHTML=`⟨a,b⟩ = ${dot.toFixed(2)} = ‖a‖·‖b‖·cosφ = ${na.toFixed(2)}·${nb.toFixed(2)}·${cos.toFixed(2)} · φ≈${ang.toFixed(0)}° · ${sign}`;}
 R();})();
"""
display(HTML(viz(body, script)))

In [ ]:
#@title Теория 2 — Нормализация (L2) { display-mode: "form" }
body = r"""
<h2>2. Длина через скалярное произведение и нормализация</h2>

<p>Как отмечено выше, $\langle a,a\rangle=\lVert a\rVert^2$, то есть евклидова норма
(«длина», $L_2$-норма) выражается через скалярное произведение:
$$ \lVert a\rVert=\sqrt{\langle a,a\rangle}=\sqrt{\sum_{i=1}^{n} a_i^2}. $$</p>

<div class="box def"><div class="t">Определение (нормализация)</div>
<b>$L_2$-нормализацией</b> ненулевого вектора $a$ называют переход к вектору
$$ \hat a=\frac{a}{\lVert a\rVert}, $$
имеющему единичную длину ($\lVert\hat a\rVert=1$) и <b>то же направление</b>, что и $a$.
Полученный $\hat a$ называют <b>единичным</b> (нормированным) вектором. Все единичные
векторы плоскости образуют окружность радиуса $1$; в $\mathbb{R}^n$ — единичную сферу.</div>

<p>Нормализация «стирает» длину, сохраняя направление: остаётся только информация о
<em>соотношении</em> координат, но не об их абсолютной величине. Перетащите $a$ и
проследите, как нормированный $\hat a$ скользит по единичной окружности.</p>

<div class="widget"><div class="widget-title">$L_2$-нормализация: проекция на единичную окружность</div>
<div class="widget-sub">Перетащите $a$ (янтарный). Зелёный $\hat a=a/\lVert a\rVert$ всегда лежит на окружности.</div>
<svg viewBox="0 0 600 380" id="svg-nrm"></svg>
<div class="readout" id="nrm-ro"></div></div>

<h3>Две разные операции со словом «нормализация»</h3>
<p>Здесь важно не смешивать два действия, которые по-русски часто называют одинаково.</p>
<ul>
<li><b>Нормализация вектора ($L_2$, по строке)</b> — определённая выше $\hat a=a/\lVert a\rVert$.
Она делит <em>все координаты одного наблюдения</em> на общее число — его длину. Соотношение
признаков внутри строки при этом не меняется, меняется лишь длина. Нужна для косинусной
близости и сравнения эмбеддингов.</li>
<li><b>Масштабирование признаков (по столбцу)</b> — приведение <em>каждого столбца</em> к
сравнимому масштабу (например, вычесть среднее и поделить на стандартное отклонение —
$z$-оценка). Нужна, чтобы ни один признак не доминировал в мерах близости только из-за
единиц измерения.</li>
</ul>
<p>Первое <b>не заменяет</b> второго: деля строку на её длину, мы масштабируем все признаки
одинаково, поэтому «крупный» по величине признак как подавлял остальные, так и подавляет.
Рассмотрим пример.</p>

<div class="box warn"><div class="t">Пример: без масштабирования вклад признака подавляется</div>
<p>Три автомобиля; признаки — марка (one-hot: $0$ — Toyota, $1$ — BMW), пробег (км) и цена:</p>
<table>
<tr><th>авто</th><th>марка</th><th>пробег, км</th><th>цена</th></tr>
<tr><td>A</td><td>0</td><td>50 000</td><td>1 000 000</td></tr>
<tr><td>B</td><td>1</td><td>50 000</td><td>1 000 000</td></tr>
<tr><td>C</td><td>0</td><td>52 000</td><td>1 020 000</td></tr>
</table>
<p>Найдём евклидовы расстояния до A <b>без масштабирования</b>:
$$ d(A,B)=\sqrt{1^2+0+0}=1,\qquad d(A,C)=\sqrt{0+2000^2+20000^2}\approx 20100. $$
Расстояние целиком определяется ценой и пробегом, а смена марки даёт вклад ровно $1$ — на их
фоне он <b>незаметен</b>. Формально ближайшим к A оказывается B (другой марки!), а C — «далёким»,
хотя различие в цене на $20\,000$ при цене в миллион ничтожно. Признак «марка» на своей шкале
$\{0,1\}$ на близость не влияет никак.</p></div>

<p>Эту проблему устраняет <b>масштабирование по столбцам</b>: приведя каждый признак к единому
масштабу, мы делаем смену марки сопоставимой с типичными различиями в цене и пробеге, и она
начинает учитываться. Это касается <em>и</em> евклидова расстояния, <em>и</em> косинуса:
скалярное произведение $\sum_i a_ib_i$ точно так же определяется преимущественно признаком с
наибольшим масштабом.
Само по себе $L_2$-нормирование строки эту проблему не решает — оно про направление, а не про
соизмеримость признаков.</p>

<details><summary>Почему $L_2$-нормализация строки здесь не помогает</summary>
<p>Обе операции действуют вдоль разных осей таблицы данных, где строки — наблюдения, а
столбцы — признаки. $L_2$-нормализация делит каждую строку на её собственную длину, поэтому
соотношение признаков внутри строки сохраняется, и признак, крупный по абсолютной величине,
остаётся крупным. Применим её к тем же автомобилям:
$$ \hat a_A=(0,\ 0{,}050,\ 0{,}999),\quad
   \hat a_B=(0,\ 0{,}050,\ 0{,}999),\quad
   \hat a_C=(0,\ 0{,}051,\ 0{,}999). $$
Цена ($\approx 0{,}999$) по-прежнему подавляет пробег ($\approx 0{,}05$) и марку ($0$); все
три вектора почти совпадают, и косинусная близость между любыми двумя равна $1{,}000$ —
различить автомобили невозможно.</p>
<p>Масштабирование признаков делит каждый <em>столбец</em> на его стандартное отклонение,
вычисленное по всем наблюдениям, и лишь тогда вклад марки становится соизмерим с ценой и
пробегом. Итог:</p>
<table>
<tr><th>операция</th><th>ось</th><th>делит на</th><th>что устраняет</th></tr>
<tr><td>$L_2$-нормализация</td><td>строка (наблюдение)</td><td>длину строки $\lVert x\rVert$</td><td>общий размер наблюдения</td></tr>
<tr><td>масштабирование признаков</td><td>столбец (признак)</td><td>разброс столбца $\sigma$</td><td>несоизмеримость шкал признаков</td></tr>
</table>
</details>

<div class="box idea"><div class="t">Итог различения</div>
<b>$L_2$-нормализация (по строке)</b> убирает длину наблюдения — нужна для косинуса и
эмбеддингов. <b>Масштабирование признаков (по столбцу, $z$-оценка)</b> уравнивает вклад
признаков — нужно, чтобы близость не определялась одной «крупной» колонкой. Про $z$-оценку
подробно — в статистическом треке (неделя 1).</div>
"""
script = r"""
(function(){const svg=document.getElementById("svg-nrm");if(!svg)return;
 const ro=document.getElementById("nrm-ro");const P=plane(2.6);let a={x:1.9,y:1.2};
 function R(){while(svg.firstChild)svg.removeChild(svg.firstChild);const g=el("g");svg.appendChild(g);P.frame(g);
  g.appendChild(el("circle",{cx:P.ox,cy:P.oy,r:P.ppu,fill:"none",stroke:"#3a434f","stroke-width":1.4,"stroke-dasharray":"4 4"}));
  const n=Math.hypot(a.x,a.y)||1e-9;const h={x:a.x/n,y:a.y/n};
  g.appendChild(el("line",{x1:P.ox,y1:P.oy,x2:P.sx(a.x),y2:P.sy(a.y),stroke:COL.b,"stroke-width":1.4,"stroke-dasharray":"2 3"}));
  g.appendChild(arrow(P.ox,P.oy,P.sx(h.x),P.sy(h.y),{color:COL.res,width:3}));
  g.appendChild(arrow(P.ox,P.oy,P.sx(a.x),P.sy(a.y),{color:COL.b,width:2.2}));
  P.drag(svg,g,a.x,a.y,COL.b,(x,y)=>{a={x,y};R();});
  const t1=el("text",{x:P.sx(a.x)+6,y:P.sy(a.y),fill:COL.b,"font-family":"sans-serif","font-size":15,"font-weight":700});t1.textContent="a";g.appendChild(t1);
  const t2=el("text",{x:P.sx(h.x)+6,y:P.sy(h.y)-4,fill:COL.res,"font-family":"sans-serif","font-size":15,"font-weight":700});t2.textContent="â";g.appendChild(t2);
  ro.innerHTML=`‖a‖ = ${n.toFixed(3)} · â = (${h.x.toFixed(3)}, ${h.y.toFixed(3)}) · ‖â‖ = ${Math.hypot(h.x,h.y).toFixed(3)}`;}
 R();})();
"""
display(HTML(viz(body, script)))

In [ ]:
#@title Теория 3 — Косинусная близость vs евклидово расстояние { display-mode: "form" }
body = r"""
<h2>3. Косинусная близость и косинусное расстояние</h2>

<p>Из геометрической формы скалярного произведения выражается косинус угла между векторами:
$$ \cos\varphi=\frac{\langle a,b\rangle}{\lVert a\rVert\,\lVert b\rVert}. $$
Эту величину называют <b>косинусной близостью</b> (cosine similarity). Она лежит в
пределах $[-1,1]$: значение $1$ — сонаправленные векторы, $0$ — ортогональные, $-1$ —
противоположно направленные. Соответствующее <b>косинусное расстояние</b> определяют как
$$ d_{\cos}(a,b)=1-\cos\varphi. $$</p>

<div class="box def"><div class="t">Ключевое отличие от евклидова расстояния</div>
Евклидово расстояние $\lVert a-b\rVert$ учитывает и направление, и <b>длину</b> векторов;
косинусная близость зависит <b>только от направления</b> и не меняется при умножении любого
из векторов на положительное число. Поэтому косинус отвечает на вопрос «насколько похожи
<em>соотношения</em> признаков», а не «насколько близки сами величины».</div>

<p>Связь двух мер становится точной <b>после нормализации</b>. Для единичных векторов
$\hat a,\hat b$ прямым раскрытием квадрата нормы получаем
$$ \lVert\hat a-\hat b\rVert^2=\langle\hat a-\hat b,\ \hat a-\hat b\rangle
   =2-2\langle\hat a,\hat b\rangle=2\,(1-\cos\varphi). $$
То есть на единичной окружности евклидово расстояние — это <b>монотонная функция</b>
косинусного: они упорядочивают пары одинаково. Вот почему для нормированных векторов
косинусная близость показательна — она измеряет ровно то же, что и расстояние по прямой,
но не зависит от исходных длин.</p>

<div class="widget"><div class="widget-title">Евклид против косинуса: включите нормализацию</div>
<div class="widget-sub">Перетащите $a,b$. Пунктирная хорда — евклидово расстояние. Флажок
переносит векторы на единичную окружность (нормализация); угол при этом не меняется.</div>
<svg viewBox="0 0 600 380" id="svg-ec"></svg>
<div class="controls"><div class="control">
<label><input type="checkbox" id="ec-norm"> нормализовать (перенести на окружность)</label></div></div>
<div class="readout" id="ec-ro"></div></div>

<p>Обратите внимание: при нормализации <b>косинус не меняется</b> (угол тот же), а евклидово
расстояние меняется и становится согласованным с косинусным по формуле выше.</p>

<div class="box warn"><div class="t">Осторожно с неотрицательными признаками</div>
Если все признаки положительны (как вес и рост), все векторы лежат в одном квадранте, углы
малы, и косинусная близость почти всегда близка к $1$ — различия слабые. Чтобы косинус стал
информативным, данные обычно предварительно <b>центрируют</b> (вычитают среднее). Мы
вернёмся к центрированию при изучении ковариации и PCA.</div>
"""
script = r"""
(function(){const svg=document.getElementById("svg-ec");if(!svg)return;
 const ro=document.getElementById("ec-ro"),cb=document.getElementById("ec-norm");
 const P=plane(2.6);let a={x:2.0,y:0.7},b={x:0.8,y:1.9};
 function R(){while(svg.firstChild)svg.removeChild(svg.firstChild);const g=el("g");svg.appendChild(g);P.frame(g);
  g.appendChild(el("circle",{cx:P.ox,cy:P.oy,r:P.ppu,fill:"none",stroke:"#2a313b","stroke-width":1.3,"stroke-dasharray":"4 4"}));
  const na=Math.hypot(a.x,a.y)||1e-9,nb=Math.hypot(b.x,b.y)||1e-9,norm=cb.checked;
  const A=norm?{x:a.x/na,y:a.y/na}:a,B=norm?{x:b.x/nb,y:b.y/nb}:b;
  // raw thin references
  g.appendChild(arrow(P.ox,P.oy,P.sx(a.x),P.sy(a.y),{color:COL.a,dash:"2 3",width:1.2}));
  g.appendChild(arrow(P.ox,P.oy,P.sx(b.x),P.sy(b.y),{color:COL.b,dash:"2 3",width:1.2}));
  // euclid chord
  g.appendChild(el("line",{x1:P.sx(A.x),y1:P.sy(A.y),x2:P.sx(B.x),y2:P.sy(B.y),stroke:COL.sum,"stroke-width":2,"stroke-dasharray":"5 4"}));
  g.appendChild(arrow(P.ox,P.oy,P.sx(A.x),P.sy(A.y),{color:COL.a,width:2.6}));
  g.appendChild(arrow(P.ox,P.oy,P.sx(B.x),P.sy(B.y),{color:COL.b,width:2.6}));
  P.drag(svg,g,a.x,a.y,COL.a,(x,y)=>{a={x,y};R();});
  P.drag(svg,g,b.x,b.y,COL.b,(x,y)=>{b={x,y};R();});
  const dot=a.x*b.x+a.y*b.y,cos=dot/(na*nb),eu=Math.hypot(A.x-B.x,A.y-B.y);
  let s=`евклид = ${eu.toFixed(3)} · cos = ${cos.toFixed(3)} · cos-dist = ${(1-cos).toFixed(3)}`;
  if(norm)s+=` · √(2(1−cos)) = ${Math.sqrt(Math.max(0,2*(1-cos))).toFixed(3)}`;
  ro.innerHTML=s;}
 cb.addEventListener("change",R);R();})();
"""
display(HTML(viz(body, script)))

In [ ]:
#@title Теория 4 — Где это в ML: эмбеддинги и внимание { display-mode: "form" }
body = r"""
<h2>4. Где это работает в машинном обучении</h2>

<h3>Эмбеддинги и поиск похожего</h3>
<p>Современные модели превращают слова, тексты, изображения и товары в <b>эмбеддинги</b> —
векторы в пространстве $\mathbb{R}^n$ высокой размерности ($n$ порядка сотен и тысяч),
устроенные так, что близкие по смыслу объекты получают близкие по <em>направлению</em>
векторы. Стандартная мера похожести здесь — <b>косинусная близость</b>: поиск похожих
документов, рекомендации, кластеризация текстов сводятся к отысканию векторов с наибольшим
$\cos\varphi$ к запросу. Длина эмбеддинга часто малоинформативна, поэтому берут именно
косинус, а не евклидово расстояние.</p>

<h3>Механизм внимания (attention) в языковых моделях</h3>
<p>В основе трансформеров — тех самых моделей, что стоят за большими языковыми моделями, —
лежит скалярное произведение. Каждому элементу последовательности сопоставляют три вектора:
<b>запрос</b> $q$ (query), <b>ключ</b> $k$ (key) и <b>значение</b> $v$ (value). Насколько
один элемент должен «обратить внимание» на другой, определяется <b>скалярным произведением</b>
запроса и ключа:
$$ \text{score}(q,k)=\frac{\langle q,k\rangle}{\sqrt{d}}, $$
где $d$ — размерность (деление на $\sqrt{d}$ стабилизирует масштаб). Полученные оценки
пропускают через $\operatorname{softmax}$, получая веса $w_i\ge 0$, $\sum_i w_i=1$, и
итоговый вектор есть взвешенная сумма значений $\sum_i w_i v_i$. Иными словами, чем сильнее
запрос сонаправлен с ключом, тем больший вес получает соответствующее значение.</p>

<p>Ниже — упрощённая модель внимания: перетаскивайте <b>запрос</b> $q$ (белый) и смотрите,
как перераспределяются веса между тремя ключами и куда смещается результат (зелёная точка).</p>

<div class="widget"><div class="widget-title">Внимание как softmax по скалярным произведениям</div>
<div class="widget-sub">Перетащите $q$. Веса $w_i=\operatorname{softmax}(\langle q,k_i\rangle/\sqrt2)$;
результат — $\sum_i w_i k_i$.</div>
<svg viewBox="0 0 600 380" id="svg-att"></svg>
<div class="readout" id="att-ro"></div></div>

<div class="box take"><div class="t">Главная мысль урока</div>
Скалярное произведение — это способ измерить «сколько одного направления содержится в
другом». Из него следуют длина ($\langle a,a\rangle$), угол и косинусная близость, а на
нормированных векторах косинус согласован с евклидовым расстоянием. Эта же операция
управляет вниманием в языковых моделях — поэтому она одна из самых востребованных в ML.</div>
"""
script = r"""
(function(){const svg=document.getElementById("svg-att");if(!svg)return;
 const ro=document.getElementById("att-ro");
 const keys=[{x:3.6,y:1.3,n:"k₁"},{x:-2.6,y:2.4,n:"k₂"},{x:-1.2,y:-2.8,n:"k₃"}];
 const cols=["#6ab0f3","#e0b25a","#e0757f"];let q={x:2.9,y:1.7};
 function R(){const g=freshFrame(svg);const d=Math.sqrt(2);
  const s=keys.map(k=>(q.x*k.x+q.y*k.y)/d);const m=Math.max(...s);
  const ex=s.map(v=>Math.exp(v-m));const Z=ex.reduce((p,c)=>p+c,0);const w=ex.map(v=>v/Z);
  const c={x:keys.reduce((p,k,i)=>p+w[i]*k.x,0),y:keys.reduce((p,k,i)=>p+w[i]*k.y,0)};
  keys.forEach((k,i)=>{g.appendChild(arrow(OX,OY,SX(k.x),SY(k.y),{color:cols[i],width:1.6+3*w[i]}));
    label(g,k.x+.15,k.y+.3,k.n,cols[i]);});
  g.appendChild(el("circle",{cx:SX(c.x),cy:SY(c.y),r:6,fill:"#5fd08a"}));
  label(g,c.x+.2,c.y-.2,"итог",'#5fd08a');
  g.appendChild(arrow(OX,OY,SX(q.x),SY(q.y),{color:"#ffffff",width:3}));
  handle(g,q.x,q.y,"#ffffff",(x,y)=>{q={x,y};R();},svg);
  label(g,q.x+.2,q.y+.3,"q",'#ffffff');
  ro.innerHTML="softmax-веса: "+w.map((v,i)=>`<span style="color:${cols[i]}">${keys[i].n} = ${v.toFixed(2)}</span>`).join("  ·  ");}
 R();})();
"""
display(HTML(viz(body, script)))

## Practice

We work with the `weight` and `height` columns as 2-D observation vectors.
Fill in the `# TODO` lines. Do each computation **by hand with numpy first**, then
cross-check against the library helper.

In [ ]:
# Observation matrix: each row is a vector (weight, height)
X = df[['weight', 'height']].to_numpy(float)
X.shape   # (200, 2)

### Task 1 — dot product two ways

Take observations 0 and 1. Compute the dot product $\langle a,b\rangle$ two ways:
elementwise `(a*b).sum()` and via the operator `a @ b`. They must match.
Then verify the identity $\langle a,a\rangle = \lVert a\rVert^2$.

In [ ]:
a = X[0]
b = X[1]

# TODO: dot product as elementwise sum: (a * b).sum()
# TODO: dot product via the @ operator: a @ b
# TODO: check that a @ a equals np.linalg.norm(a) ** 2


### Task 2 — cosine similarity

Implement cosine similarity $\cos\varphi = \dfrac{\langle a,b\rangle}{\lVert a\rVert\,\lVert b\rVert}$
as a function, then apply it to observations 0 and 1. Also report the angle in degrees
(`np.degrees(np.arccos(...))`).

In [ ]:
def cosine_similarity(u, v):
    # TODO: return (u @ v) / (norm(u) * norm(v))
    ...

# TODO: cosine_similarity(X[0], X[1]) and the angle in degrees


### Task 3 — magnitude vs direction

Cosine ignores length; euclidean distance does not. Build a vector with the **same
direction** but a larger magnitude and observe the two measures disagree.

In [ ]:
p = X[0]
p_scaled = 3.0 * p          # same direction, three times longer

# TODO: euclidean distance np.linalg.norm(p - p_scaled)  -> large
# TODO: cosine_similarity(p, p_scaled)                    -> 1.0 (identical direction)


### Task 4 — nearest neighbour: euclidean vs cosine

For observation 0, find its nearest neighbour under **euclidean** distance and under
**cosine** distance, and compare. Because weight/height are positive, first **center** the
data (subtract the mean) so that cosine becomes informative.

In [ ]:
Xc = X - X.mean(axis=0)                       # center the cloud
Xn = Xc / np.linalg.norm(Xc, axis=1, keepdims=True)   # L2-normalize the centered rows

# TODO: euclidean nearest neighbour of row 0
#   d_euc = np.linalg.norm(Xc - Xc[0], axis=1); d_euc[0] = np.inf; d_euc.argmin()
# TODO: cosine nearest neighbour of row 0 (largest cosine = smallest cosine distance)
#   sims = Xn @ Xn[0]; sims[0] = -np.inf; sims.argmax()
# TODO: are the two neighbours the same row? print both


### Homework

1. Write a function `cosine_distance(u, v)` returning $1-\cos\varphi$ and confirm it is
   `0` for a vector with itself and `2` for a vector and its negation.
2. Using the **centered, L2-normalized** matrix `Xn`, verify the identity
   $\lVert \hat a-\hat b\rVert^2 = 2\,(1-\cos\varphi)$ on a few random pairs.
3. Pick two people with a similar height-to-weight ratio but different overall size.
   Show numerically that their **cosine** similarity is high while their **euclidean**
   distance is not small. Explain in one sentence what this means for the data.
4. *(Embeddings framing.)* Explain in two or three sentences why cosine similarity, rather
   than euclidean distance, is the default measure for comparing high-dimensional
   embeddings, and how this reuses the dot-product identity from this lesson.

---
### Итог урока

- **Скалярное произведение** $\langle a,b\rangle=\sum_i a_ib_i=\lVert a\rVert\lVert b\rVert\cos\varphi$ — число, измеряющее сонаправленность.
- Из него следуют длина ($\langle a,a\rangle=\lVert a\rVert^2$), проекция и угол.
- **$L_2$-нормализация** $\hat a=a/\lVert a\rVert$ убирает длину, оставляя направление.
- **Косинусная близость** зависит только от направления; на нормированных векторах согласована с евклидовым расстоянием: $\lVert\hat a-\hat b\rVert^2=2(1-\cos\varphi)$.
- Эта операция — основа сравнения эмбеддингов и механизма внимания в языковых моделях.
